# AWS S3 Data Intelligence Lab
## Cortex AI Functions + Search + Analyst + Agent

**What you'll build:** A complete pipeline that auto-ingests files from S3, enriches them with 11 Cortex AI functions, and exposes everything through a unified Cortex Agent accessible via Snowflake Intelligence.

**Duration:** ~90 minutes

### Before starting this notebook:
1. Create an S3 bucket with prefixes: `healthcare/pdfs/`, `healthcare/txt/`, `healthcare/audio/`
2. Create an IAM role for Snowflake (see LAB_GUIDE.md Steps 2-3)
3. Upload the sample files from `sample_files/` to your S3 bucket

---

## Step 1: Database, Schemas & Warehouse

Creates the core structure:
- **RAW** — file metadata landing zone (Snowpipe target)
- **PROCESSED** — AI-enriched intelligence tables
- **ANALYTICS** — structured data, semantic view, agent

In [ ]:
-----------------------------------------------------------------------
-- 1. DATABASE
-----------------------------------------------------------------------
CREATE OR REPLACE DATABASE HEALTHCARE_AI_DEMO
  COMMENT = 'Healthcare AI Intelligence Pipeline: PDF, TXT, and audio files processed with 11 Cortex AI functions';

-----------------------------------------------------------------------
-- 2. SCHEMAS
-----------------------------------------------------------------------
CREATE SCHEMA IF NOT EXISTS HEALTHCARE_AI_DEMO.RAW
  COMMENT = 'Landing zone for raw file metadata from S3 via Snowpipe';

CREATE SCHEMA IF NOT EXISTS HEALTHCARE_AI_DEMO.PROCESSED
  COMMENT = 'AI-enriched intelligence tables — one per file type (PDF, TXT, Audio)';

CREATE SCHEMA IF NOT EXISTS HEALTHCARE_AI_DEMO.ANALYTICS
  COMMENT = 'Structured data, analytics views, semantic view, Cortex Search, and Agent';

-----------------------------------------------------------------------
-- 3. WAREHOUSE
-----------------------------------------------------------------------
CREATE OR REPLACE WAREHOUSE HEALTHCARE_AI_WH
  WAREHOUSE_SIZE      = 'XSMALL'
  AUTO_SUSPEND        = 60
  AUTO_RESUME         = TRUE
  INITIALLY_SUSPENDED = TRUE
  COMMENT = 'Warehouse for Healthcare AI pipeline processing';


-----------------------------------------------------------------------
-- 4. VERIFY
-----------------------------------------------------------------------
SHOW SCHEMAS IN DATABASE HEALTHCARE_AI_DEMO;

## Step 2: Storage Integration

Establishes trust between Snowflake and AWS. Replace the two placeholders below with your actual values:
- `<YOUR_AWS_IAM_ROLE_ARN>` — your IAM role ARN
- `<YOUR_BUCKET_NAME>` — your S3 bucket name

**Do NOT re-run this cell** after updating the trust policy — it generates a new external ID each time.

In [ ]:
-----------------------------------------------------------------------
-- 1. STORAGE INTEGRATION
-----------------------------------------------------------------------
CREATE OR REPLACE STORAGE INTEGRATION HEALTHCARE_S3_INTEGRATION
  TYPE                      = EXTERNAL_STAGE
  STORAGE_PROVIDER          = 'S3'
  ENABLED                   = TRUE
  STORAGE_AWS_ROLE_ARN      = '<YOUR_AWS_IAM_ROLE_ARN>'
  STORAGE_ALLOWED_LOCATIONS = ('s3://<YOUR_S3_BUCKET_NAME>/healthcare/')
  COMMENT = 'Storage integration for Healthcare AI demo S3 bucket';

-- Get Snowflake IAM user ARN and external ID for AWS trust policy:
DESCRIBE INTEGRATION HEALTHCARE_S3_INTEGRATION;
-- Record: STORAGE_AWS_IAM_USER_ARN and STORAGE_AWS_EXTERNAL_ID
-- Update your IAM role trust policy with these values (see 14_aws_setup_guide.sql)

-----------------------------------------------------------------------
-- 2. GRANT USAGE (if using roles other than ACCOUNTADMIN)
-----------------------------------------------------------------------
GRANT USAGE ON INTEGRATION HEALTHCARE_S3_INTEGRATION TO ROLE SYSADMIN;

-----------------------------------------------------------------------
-- 3. EXTERNAL STAGES (one per file type prefix in S3)
-----------------------------------------------------------------------
CREATE OR REPLACE STAGE RAW.S3_MEDICAL_DOCS
  URL                 = 's3://<YOUR_S3_BUCKET_NAME>/healthcare/pdfs/'
  STORAGE_INTEGRATION = HEALTHCARE_S3_INTEGRATION
  DIRECTORY           = (ENABLE = TRUE AUTO_REFRESH = TRUE)
  COMMENT = 'S3 stage for incoming medical PDF documents';

CREATE OR REPLACE STAGE RAW.S3_MEDICAL_TXT
  URL                 = 's3://<YOUR_S3_BUCKET_NAME>/healthcare/txt/'
  STORAGE_INTEGRATION = HEALTHCARE_S3_INTEGRATION
  DIRECTORY           = (ENABLE = TRUE AUTO_REFRESH = TRUE)
  COMMENT = 'S3 stage for incoming medical text documents';

CREATE OR REPLACE STAGE RAW.S3_MEDICAL_AUDIO
  URL                 = 's3://<YOUR_S3_BUCKET_NAME>/healthcare/audio/'
  STORAGE_INTEGRATION = HEALTHCARE_S3_INTEGRATION
  DIRECTORY           = (ENABLE = TRUE AUTO_REFRESH = TRUE)
  COMMENT = 'S3 stage for incoming patient consultation audio files (WAV and MP3)';

-----------------------------------------------------------------------
-- 4. INTERNAL STAGE (for semantic model YAML)
-----------------------------------------------------------------------
CREATE OR REPLACE STAGE ANALYTICS.SEMANTIC_MODELS
  DIRECTORY = (ENABLE = TRUE)
  COMMENT = 'Internal stage for semantic model YAML files';

-----------------------------------------------------------------------
-- 5. VERIFY
-----------------------------------------------------------------------
SHOW INTEGRATIONS LIKE 'HEALTHCARE%';
SHOW STAGES IN DATABASE HEALTHCARE_AI_DEMO;

### Get Snowflake IAM User ARN & External ID

Copy `STORAGE_AWS_IAM_USER_ARN` and `STORAGE_AWS_EXTERNAL_ID` from the output below.

**Then update your IAM role trust policy in AWS Console** with these values. Wait 30 seconds before continuing.

In [ ]:
DESCRIBE INTEGRATION HEALTHCARE_S3_INTEGRATION;

### Verify S3 Access (should return files or empty without errors)

In [ ]:
LIST @HEALTHCARE_AI_DEMO.RAW.S3_MEDICAL_DOCS;

## Step 3: Snowpipes & Stream

Snowpipe with `AUTO_INGEST = TRUE` creates a managed SQS queue. We only ingest **file metadata** — actual files stay in S3 and are read on-demand by AI functions via `TO_FILE()`.

In [ ]:
-----------------------------------------------------------------------
-- 1. FILES_LOG TABLE -- every file that lands in S3 gets a row here
-----------------------------------------------------------------------
CREATE OR REPLACE TABLE RAW.FILES_LOG (
    FILE_ID         NUMBER AUTOINCREMENT PRIMARY KEY,
    FILE_NAME       VARCHAR        NOT NULL,
    FILE_PATH       VARCHAR        NOT NULL,
    FILE_TYPE       VARCHAR(10)    NOT NULL COMMENT 'PDF, TXT, WAV, or MP3',
    S3_EVENT_TIME   TIMESTAMP_NTZ,
    LANDED_AT       TIMESTAMP_NTZ  DEFAULT CURRENT_TIMESTAMP(),
    IS_PROCESSED    BOOLEAN        DEFAULT FALSE,
    PROCESSED_AT    TIMESTAMP_NTZ
);

-----------------------------------------------------------------------
-- 2. FILE FORMAT
--    We only need METADATA$ columns (file name, scan time), not file
--    content. CSV with no delimiters avoids parse errors on binary
--    (PDF, WAV, MP3) and plain text files.
-----------------------------------------------------------------------
CREATE OR REPLACE FILE FORMAT RAW.METADATA_ONLY_FORMAT
  TYPE             = 'CSV'
  RECORD_DELIMITER = NONE
  FIELD_DELIMITER  = NONE
  COMMENT = 'Format for metadata-only ingestion -- skips file content parsing';

-----------------------------------------------------------------------
-- 3. SNOWPIPE FOR PDFs
-----------------------------------------------------------------------
CREATE OR REPLACE PIPE RAW.PIPE_MEDICAL_DOCS
  AUTO_INGEST = TRUE
  COMMENT = 'Auto-ingest pipe for PDF medical documents from S3 via SQS'
AS
  COPY INTO RAW.FILES_LOG (FILE_NAME, FILE_PATH, FILE_TYPE, S3_EVENT_TIME)
  FROM (
    SELECT
      METADATA$FILENAME            AS FILE_NAME,
      METADATA$FILENAME            AS FILE_PATH,
      'PDF'                        AS FILE_TYPE,
      METADATA$START_SCAN_TIME     AS S3_EVENT_TIME
    FROM @RAW.S3_MEDICAL_DOCS
  )
  FILE_FORMAT = (FORMAT_NAME = 'RAW.METADATA_ONLY_FORMAT');

-----------------------------------------------------------------------
-- 4. SNOWPIPE FOR TXT
-----------------------------------------------------------------------
CREATE OR REPLACE PIPE RAW.PIPE_MEDICAL_TXT
  AUTO_INGEST = TRUE
  COMMENT = 'Auto-ingest pipe for TXT medical documents from S3 via SQS'
AS
  COPY INTO RAW.FILES_LOG (FILE_NAME, FILE_PATH, FILE_TYPE, S3_EVENT_TIME)
  FROM (
    SELECT
      METADATA$FILENAME            AS FILE_NAME,
      METADATA$FILENAME            AS FILE_PATH,
      'TXT'                        AS FILE_TYPE,
      METADATA$START_SCAN_TIME     AS S3_EVENT_TIME
    FROM @RAW.S3_MEDICAL_TXT
  )
  FILE_FORMAT = (FORMAT_NAME = 'RAW.METADATA_ONLY_FORMAT');

-----------------------------------------------------------------------
-- 5. SNOWPIPE FOR AUDIO (WAV and MP3)
-----------------------------------------------------------------------
CREATE OR REPLACE PIPE RAW.PIPE_MEDICAL_AUDIO
  AUTO_INGEST = TRUE
  COMMENT = 'Auto-ingest pipe for WAV/MP3 audio consultations from S3 via SQS'
AS
  COPY INTO RAW.FILES_LOG (FILE_NAME, FILE_PATH, FILE_TYPE, S3_EVENT_TIME)
  FROM (
    SELECT
      METADATA$FILENAME            AS FILE_NAME,
      METADATA$FILENAME            AS FILE_PATH,
      CASE
        WHEN METADATA$FILENAME ILIKE '%.mp3' THEN 'MP3'
        ELSE 'WAV'
      END                          AS FILE_TYPE,
      METADATA$START_SCAN_TIME     AS S3_EVENT_TIME
    FROM @RAW.S3_MEDICAL_AUDIO
  )
  FILE_FORMAT = (FORMAT_NAME = 'RAW.METADATA_ONLY_FORMAT');

-----------------------------------------------------------------------
-- 6. GET SQS QUEUE ARNs
--    The notification_channel column contains the Snowflake-managed
--    SQS queue ARN. All pipes share the same ARN per account.
--    Copy this ARN for S3 event notification setup.
-----------------------------------------------------------------------
SHOW PIPES IN SCHEMA RAW;

-- Per-pipe details:
DESC PIPE RAW.PIPE_MEDICAL_DOCS;
DESC PIPE RAW.PIPE_MEDICAL_TXT;
DESC PIPE RAW.PIPE_MEDICAL_AUDIO;

-----------------------------------------------------------------------
-- 7. STREAM on FILES_LOG -- captures new inserts for processing
-----------------------------------------------------------------------
CREATE OR REPLACE STREAM RAW.FILES_LOG_STREAM
  ON TABLE RAW.FILES_LOG
  APPEND_ONLY = TRUE
  COMMENT = 'Captures new file arrivals for AI processing';

-----------------------------------------------------------------------
-- 8. VERIFY
-----------------------------------------------------------------------
SHOW PIPES IN SCHEMA RAW;
SHOW STREAMS IN SCHEMA RAW;

SELECT SYSTEM$PIPE_STATUS('RAW.PIPE_MEDICAL_DOCS')  AS DOCS_PIPE_STATUS;
SELECT SYSTEM$PIPE_STATUS('RAW.PIPE_MEDICAL_TXT')   AS TXT_PIPE_STATUS;
SELECT SYSTEM$PIPE_STATUS('RAW.PIPE_MEDICAL_AUDIO') AS AUDIO_PIPE_STATUS;

### Get SQS Queue ARN

Copy the `notification_channel` value from any pipe row. Use it to configure S3 event notifications.

---
### ⏸️ PAUSE: Configure S3 Event Notifications

Go to **AWS Console → S3 → Your bucket → Properties → Event notifications** and create 3 notifications pointing to the SQS ARN:
- `snowpipe-pdfs` → prefix `healthcare/pdfs/`
- `snowpipe-txt` → prefix `healthcare/txt/`
- `snowpipe-audio` → prefix `healthcare/audio/`

Continue below once saved.

In [ ]:
SHOW PIPES IN SCHEMA HEALTHCARE_AI_DEMO.RAW;

## Step 4: Trigger Ingestion & Verify

Refresh pipes to load files already in S3. Wait 1-2 minutes after running.

In [ ]:
ALTER PIPE HEALTHCARE_AI_DEMO.RAW.PIPE_MEDICAL_DOCS REFRESH;
ALTER PIPE HEALTHCARE_AI_DEMO.RAW.PIPE_MEDICAL_TXT REFRESH;
ALTER PIPE HEALTHCARE_AI_DEMO.RAW.PIPE_MEDICAL_AUDIO REFRESH;

### Verify (expected: 6 PDF, 4 TXT, 5 WAV/MP3)

In [ ]:
SELECT FILE_TYPE, COUNT(*) AS FILES FROM HEALTHCARE_AI_DEMO.RAW.FILES_LOG GROUP BY FILE_TYPE;

### Fix FILE_NAME prefix

Snowpipe stores the full S3 path. AI functions need just the filename relative to the stage.

In [ ]:
UPDATE HEALTHCARE_AI_DEMO.RAW.FILES_LOG
SET FILE_NAME = REGEXP_REPLACE(FILE_NAME, '^healthcare/(pdfs|txt|audio)/', '')
WHERE FILE_NAME LIKE 'healthcare/%';

## Step 5: AI Processing Tables

One table per file type to store the output of all 9-10 AI functions.

In [ ]:
-----------------------------------------------------------------------
-- 1. PDF_INTELLIGENCE — one row per processed PDF
-----------------------------------------------------------------------
CREATE OR REPLACE TABLE PROCESSED.PDF_INTELLIGENCE (
    DOC_ID                  NUMBER AUTOINCREMENT PRIMARY KEY,
    FILE_ID                 NUMBER         NOT NULL COMMENT 'FK to RAW.FILES_LOG',
    FILE_NAME               VARCHAR        NOT NULL,
    PROCESSED_AT            TIMESTAMP_NTZ  DEFAULT CURRENT_TIMESTAMP(),

    -- AI_PARSE_DOCUMENT output (OCR + layout)
    PARSED_TEXT             VARCHAR(16777216) COMMENT 'Raw OCR text from AI_PARSE_DOCUMENT',
    PARSED_LAYOUT           VARIANT           COMMENT 'Layout-mode structured output (tables, paragraphs)',

    -- AI_EXTRACT output
    EXTRACTED_FIELDS        VARIANT           COMMENT 'Structured: patient_name, dob, diagnosis, provider, medications, amounts',

    -- AI_CLASSIFY output
    DOC_CATEGORY            VARCHAR(100)      COMMENT 'Lab Report, Discharge Summary, Prescription, Radiology Report, etc.',
    DOC_CATEGORY_CONFIDENCE FLOAT             COMMENT 'Classification confidence score',

    -- AI_SENTIMENT output
    SENTIMENT_SCORE         FLOAT             COMMENT 'Overall sentiment (-1 to 1)',
    SENTIMENT_DIMENSIONS    VARIANT           COMMENT 'Multi-dimensional: urgency, clinical_concern, patient_satisfaction',

    -- AI_SUMMARIZE output
    SUMMARY                 VARCHAR(10000)    COMMENT '3-sentence summary of the document',

    -- AI_TRANSLATE output
    DETECTED_LANGUAGE       VARCHAR(50)       COMMENT 'Detected source language',
    TRANSLATED_TEXT         VARCHAR(16777216)  COMMENT 'English translation (NULL if already English)',

    -- AI_REDACT output
    REDACTED_TEXT           VARCHAR(16777216)  COMMENT 'PII-redacted version of parsed text',

    -- AI_COMPLETE output
    KEY_INSIGHTS            VARCHAR(10000)    COMMENT 'LLM-generated key insights and action items',

    -- AI_EMBED output
    EMBEDDING               VECTOR(FLOAT, 768) COMMENT 'Text embedding for similarity search'
);

-----------------------------------------------------------------------
-- 2. TXT_INTELLIGENCE — one row per processed TXT file
-----------------------------------------------------------------------
CREATE OR REPLACE TABLE PROCESSED.TXT_INTELLIGENCE (
    TXT_ID                  NUMBER AUTOINCREMENT PRIMARY KEY,
    FILE_ID                 NUMBER         NOT NULL COMMENT 'FK to RAW.FILES_LOG',
    FILE_NAME               VARCHAR        NOT NULL,
    PROCESSED_AT            TIMESTAMP_NTZ  DEFAULT CURRENT_TIMESTAMP(),

    -- Raw text content (read directly from stage, no AI_PARSE_DOCUMENT needed)
    RAW_TEXT                VARCHAR(16777216) COMMENT 'Original text content read from the TXT file',

    -- AI_EXTRACT output
    EXTRACTED_FIELDS        VARIANT           COMMENT 'Structured: patient_name, dob, diagnosis, provider, medications, amounts',

    -- AI_CLASSIFY output
    DOC_CATEGORY            VARCHAR(100)      COMMENT 'Lab Report, Clinical Notes, Patient Intake Form, Nursing Notes, etc.',
    DOC_CATEGORY_CONFIDENCE FLOAT             COMMENT 'Classification confidence score',

    -- AI_SENTIMENT output
    SENTIMENT_SCORE         FLOAT             COMMENT 'Overall sentiment (-1 to 1)',
    SENTIMENT_DIMENSIONS    VARIANT           COMMENT 'Multi-dimensional: urgency, clinical_concern, patient_satisfaction',

    -- AI_SUMMARIZE output
    SUMMARY                 VARCHAR(10000)    COMMENT '3-sentence summary of the document',

    -- AI_TRANSLATE output
    DETECTED_LANGUAGE       VARCHAR(50)       COMMENT 'Detected source language',
    TRANSLATED_TEXT         VARCHAR(16777216)  COMMENT 'English translation (NULL if already English)',

    -- AI_REDACT output
    REDACTED_TEXT           VARCHAR(16777216)  COMMENT 'PII-redacted version of text',

    -- AI_COMPLETE output
    KEY_INSIGHTS            VARCHAR(10000)    COMMENT 'LLM-generated key insights and action items',

    -- AI_EMBED output
    EMBEDDING               VECTOR(FLOAT, 768) COMMENT 'Text embedding for similarity search'
);

-----------------------------------------------------------------------
-- 3. AUDIO_INTELLIGENCE — one row per processed WAV/MP3
-----------------------------------------------------------------------
CREATE OR REPLACE TABLE PROCESSED.AUDIO_INTELLIGENCE (
    AUDIO_ID                NUMBER AUTOINCREMENT PRIMARY KEY,
    FILE_ID                 NUMBER         NOT NULL COMMENT 'FK to RAW.FILES_LOG',
    FILE_NAME               VARCHAR        NOT NULL,
    PROCESSED_AT            TIMESTAMP_NTZ  DEFAULT CURRENT_TIMESTAMP(),

    -- AI_TRANSCRIBE output
    TRANSCRIPT_TEXT         VARCHAR(16777216) COMMENT 'Full transcription text',
    TRANSCRIPT_SEGMENTS     VARIANT           COMMENT 'Word-level timestamps and speaker diarization',
    AUDIO_DURATION_SECS     FLOAT             COMMENT 'Duration of audio in seconds',
    SPEAKER_COUNT           NUMBER            COMMENT 'Number of distinct speakers detected',

    -- AI_EXTRACT output
    EXTRACTED_FIELDS        VARIANT           COMMENT 'Structured: patient_name, provider, chief_complaint, medications, follow_ups',

    -- AI_CLASSIFY output
    CALL_CATEGORY           VARCHAR(100)      COMMENT 'Initial Consultation, Follow-Up Visit, Specialist Referral, etc.',
    CALL_CATEGORY_CONFIDENCE FLOAT            COMMENT 'Classification confidence score',

    -- AI_SENTIMENT output
    SENTIMENT_SCORE         FLOAT             COMMENT 'Overall sentiment (-1 to 1)',
    SENTIMENT_DIMENSIONS    VARIANT           COMMENT 'Multi-dimensional: empathy, clinical_clarity, patient_anxiety, resolution',

    -- AI_SUMMARIZE output
    SUMMARY                 VARCHAR(10000)    COMMENT 'Executive summary of the consultation',

    -- AI_TRANSLATE output
    DETECTED_LANGUAGE       VARCHAR(50)       COMMENT 'Detected language of the audio',
    TRANSLATED_TRANSCRIPT   VARCHAR(16777216) COMMENT 'English translation of transcript (NULL if already English)',

    -- AI_COMPLETE output
    CONSULTATION_NOTES      VARCHAR(10000)    COMMENT 'LLM-generated structured SOAP consultation notes',

    -- AI_EMBED output
    EMBEDDING               VECTOR(FLOAT, 768) COMMENT 'Transcript embedding for similarity search'
);

-----------------------------------------------------------------------
-- 4. VERIFY
-----------------------------------------------------------------------
SHOW TABLES IN SCHEMA PROCESSED;

## Step 6: AI Processing Procedures

Each procedure applies 9-10 Cortex AI functions to unprocessed files:
AI_PARSE_DOCUMENT, AI_TRANSCRIBE, AI_EXTRACT, AI_CLASSIFY, SENTIMENT, SUMMARIZE, AI_TRANSLATE, AI_REDACT, AI_COMPLETE, AI_EMBED.

> **Regional note:** If `claude-3-5-sonnet` is unavailable, replace with `mistral-large2` below.

### PDF Processing Procedure

In [ ]:
CREATE OR REPLACE PROCEDURE PROCESSED.PROCESS_PDF_FILES()
  RETURNS VARCHAR
  LANGUAGE SQL
  COMMENT = 'Processes unprocessed PDF files through 9 Cortex AI functions into PDF_INTELLIGENCE'
  EXECUTE AS CALLER
AS
BEGIN

  INSERT INTO PROCESSED.PDF_INTELLIGENCE (
      FILE_ID, FILE_NAME, PARSED_TEXT, PARSED_LAYOUT,
      EXTRACTED_FIELDS, DOC_CATEGORY, DOC_CATEGORY_CONFIDENCE,
      SENTIMENT_SCORE, SENTIMENT_DIMENSIONS, SUMMARY,
      DETECTED_LANGUAGE, TRANSLATED_TEXT, REDACTED_TEXT,
      KEY_INSIGHTS, EMBEDDING
  )
  SELECT
      f.FILE_ID,
      f.FILE_NAME,

      -----------------------------------------------------------
      -- 1. AI_PARSE_DOCUMENT (OCR) — raw text from PDF
      -----------------------------------------------------------
      AI_PARSE_DOCUMENT(
        TO_FILE('@RAW.S3_MEDICAL_DOCS', f.FILE_NAME),
        OBJECT_CONSTRUCT('mode', 'OCR')
      ):content::VARCHAR                                    AS PARSED_TEXT,

      -----------------------------------------------------------
      -- 2. AI_PARSE_DOCUMENT (LAYOUT) — structured output
      -----------------------------------------------------------
      AI_PARSE_DOCUMENT(
        TO_FILE('@RAW.S3_MEDICAL_DOCS', f.FILE_NAME),
        OBJECT_CONSTRUCT('mode', 'LAYOUT')
      )                                                     AS PARSED_LAYOUT,

      -----------------------------------------------------------
      -- 3. AI_EXTRACT — structured healthcare fields
      -----------------------------------------------------------
      AI_EXTRACT(
        AI_PARSE_DOCUMENT(
          TO_FILE('@RAW.S3_MEDICAL_DOCS', f.FILE_NAME), OBJECT_CONSTRUCT('mode', 'OCR')
        ):content::VARCHAR,
        OBJECT_CONSTRUCT(
          'patient_name',       'string: full name of the patient',
          'date_of_birth',      'string: patient date of birth',
          'provider_name',      'string: name of the doctor or provider',
          'facility_name',      'string: hospital or clinic name',
          'document_date',      'string: date of the document',
          'diagnosis',          'string: primary diagnosis or findings',
          'medications',        'array: list of medications mentioned',
          'procedures',         'array: list of procedures or tests',
          'follow_up_actions',  'array: recommended follow-up actions',
          'insurance_id',       'string: insurance or policy number if present',
          'total_amount',       'number: total billed amount if present'
        )
      )                                                     AS EXTRACTED_FIELDS,

      -----------------------------------------------------------
      -- 4. AI_CLASSIFY — categorize document type
      -----------------------------------------------------------
      AI_CLASSIFY(
        AI_PARSE_DOCUMENT(
          TO_FILE('@RAW.S3_MEDICAL_DOCS', f.FILE_NAME), OBJECT_CONSTRUCT('mode', 'OCR')
        ):content::VARCHAR,
        ARRAY_CONSTRUCT(
          'Lab Report', 'Discharge Summary', 'Prescription',
          'Radiology Report', 'Insurance Claim', 'Referral Letter',
          'Clinical Notes', 'Pathology Report'
        )
      ):labels[0]::VARCHAR                                  AS DOC_CATEGORY,

      NULL::FLOAT                                           AS DOC_CATEGORY_CONFIDENCE,

      -----------------------------------------------------------
      -- 5. AI_SENTIMENT — overall + multi-dimensional
      -----------------------------------------------------------
      SNOWFLAKE.CORTEX.SENTIMENT(
        AI_PARSE_DOCUMENT(
          TO_FILE('@RAW.S3_MEDICAL_DOCS', f.FILE_NAME), OBJECT_CONSTRUCT('mode', 'OCR')
        ):content::VARCHAR
      )                                                     AS SENTIMENT_SCORE,

      AI_SENTIMENT(
        AI_PARSE_DOCUMENT(
          TO_FILE('@RAW.S3_MEDICAL_DOCS', f.FILE_NAME), OBJECT_CONSTRUCT('mode', 'OCR')
        ):content::VARCHAR
      )                                                     AS SENTIMENT_DIMENSIONS,

      -----------------------------------------------------------
      -- 6. AI_SUMMARIZE — concise summary
      -----------------------------------------------------------
      SNOWFLAKE.CORTEX.SUMMARIZE(
        AI_PARSE_DOCUMENT(
          TO_FILE('@RAW.S3_MEDICAL_DOCS', f.FILE_NAME), OBJECT_CONSTRUCT('mode', 'OCR')
        ):content::VARCHAR
      )                                                     AS SUMMARY,

      -----------------------------------------------------------
      -- 7. AI_TRANSLATE — detect language + translate if needed
      -----------------------------------------------------------
      AI_CLASSIFY(
        AI_PARSE_DOCUMENT(
          TO_FILE('@RAW.S3_MEDICAL_DOCS', f.FILE_NAME), OBJECT_CONSTRUCT('mode', 'OCR')
        ):content::VARCHAR,
        ARRAY_CONSTRUCT('English', 'Spanish', 'French', 'German',
                        'Portuguese', 'Chinese', 'Japanese', 'Korean', 'Arabic')
      ):labels[0]::VARCHAR                                  AS DETECTED_LANGUAGE,

      CASE
        WHEN AI_CLASSIFY(
          AI_PARSE_DOCUMENT(
            TO_FILE('@RAW.S3_MEDICAL_DOCS', f.FILE_NAME), OBJECT_CONSTRUCT('mode', 'OCR')
          ):content::VARCHAR,
          ARRAY_CONSTRUCT('English', 'Spanish', 'French', 'German',
                          'Portuguese', 'Chinese', 'Japanese', 'Korean', 'Arabic')
        ):labels[0]::VARCHAR != 'English'
        THEN AI_TRANSLATE(
          AI_PARSE_DOCUMENT(
            TO_FILE('@RAW.S3_MEDICAL_DOCS', f.FILE_NAME), OBJECT_CONSTRUCT('mode', 'OCR')
          ):content::VARCHAR,
          '',
          'en'
        )
        ELSE NULL
      END                                                   AS TRANSLATED_TEXT,

      -----------------------------------------------------------
      -- 8. AI_REDACT — remove PII
      -----------------------------------------------------------
      AI_REDACT(
        AI_PARSE_DOCUMENT(
          TO_FILE('@RAW.S3_MEDICAL_DOCS', f.FILE_NAME), OBJECT_CONSTRUCT('mode', 'OCR')
        ):content::VARCHAR
      )                                                     AS REDACTED_TEXT,

      -----------------------------------------------------------
      -- 9. AI_COMPLETE — key insights and action items
      -----------------------------------------------------------
      AI_COMPLETE(
        'claude-3-5-sonnet',
        CONCAT(
          'You are a medical document analyst. Given the following medical document, ',
          'provide: 1) Three key clinical insights, 2) Any urgent action items, ',
          '3) Recommended follow-ups. Be concise.\n\nDocument:\n',
          AI_PARSE_DOCUMENT(
            TO_FILE('@RAW.S3_MEDICAL_DOCS', f.FILE_NAME), OBJECT_CONSTRUCT('mode', 'OCR')
          ):content::VARCHAR
        )
      )                                                     AS KEY_INSIGHTS,

      -----------------------------------------------------------
      -- 10. AI_EMBED — vector embedding for Cortex Search
      -----------------------------------------------------------
      AI_EMBED(
        'snowflake-arctic-embed-m-v1.5',
        AI_PARSE_DOCUMENT(
          TO_FILE('@RAW.S3_MEDICAL_DOCS', f.FILE_NAME), OBJECT_CONSTRUCT('mode', 'OCR')
        ):content::VARCHAR
      )                                                     AS EMBEDDING

  FROM RAW.FILES_LOG f
  WHERE f.FILE_TYPE = 'PDF'
    AND f.IS_PROCESSED = FALSE;

  -- Mark PDF files as processed
  UPDATE RAW.FILES_LOG
  SET IS_PROCESSED = TRUE,
      PROCESSED_AT = CURRENT_TIMESTAMP()
  WHERE FILE_TYPE = 'PDF'
    AND IS_PROCESSED = FALSE;

  RETURN 'PDF processing complete: ' || CURRENT_TIMESTAMP()::VARCHAR;

END;

### TXT Processing Procedure

In [ ]:
CREATE OR REPLACE PROCEDURE PROCESSED.PROCESS_TXT_FILES()
  RETURNS VARCHAR
  LANGUAGE SQL
  COMMENT = 'Processes unprocessed TXT files through 8 Cortex AI functions into TXT_INTELLIGENCE'
  EXECUTE AS CALLER
AS
$$
BEGIN

  INSERT INTO PROCESSED.TXT_INTELLIGENCE (
      FILE_ID, FILE_NAME, RAW_TEXT,
      EXTRACTED_FIELDS, DOC_CATEGORY, DOC_CATEGORY_CONFIDENCE,
      SENTIMENT_SCORE, SENTIMENT_DIMENSIONS, SUMMARY,
      DETECTED_LANGUAGE, TRANSLATED_TEXT, REDACTED_TEXT,
      KEY_INSIGHTS, EMBEDDING
  )
  WITH txt_content AS (
    SELECT
      f.FILE_ID,
      f.FILE_NAME,
      -- Read TXT file content via AI_PARSE_DOCUMENT (LAYOUT mode)
      AI_PARSE_DOCUMENT(
        TO_FILE('@RAW.S3_MEDICAL_TXT', f.FILE_NAME),
        OBJECT_CONSTRUCT('mode', 'LAYOUT')
      ):content::VARCHAR AS FILE_TEXT
    FROM RAW.FILES_LOG f
    WHERE f.FILE_TYPE = 'TXT'
      AND f.IS_PROCESSED = FALSE
  )
  SELECT
      t.FILE_ID,
      t.FILE_NAME,

      -- Raw text content
      t.FILE_TEXT                                            AS RAW_TEXT,

      -----------------------------------------------------------
      -- 1. AI_EXTRACT — structured healthcare fields
      -----------------------------------------------------------
      AI_EXTRACT(
        t.FILE_TEXT,
        OBJECT_CONSTRUCT(
          'patient_name',       'string: full name of the patient',
          'date_of_birth',      'string: patient date of birth',
          'provider_name',      'string: name of the doctor or provider',
          'facility_name',      'string: hospital or clinic name',
          'document_date',      'string: date of the document',
          'diagnosis',          'string: primary diagnosis or findings',
          'medications',        'array: list of medications mentioned',
          'procedures',         'array: list of procedures or tests',
          'follow_up_actions',  'array: recommended follow-up actions',
          'insurance_id',       'string: insurance or policy number if present',
          'total_amount',       'number: total billed amount if present'
        )
      )                                                     AS EXTRACTED_FIELDS,

      -----------------------------------------------------------
      -- 2. AI_CLASSIFY — categorize document type
      -----------------------------------------------------------
      AI_CLASSIFY(
        t.FILE_TEXT,
        ARRAY_CONSTRUCT(
          'Lab Report', 'Discharge Summary', 'Prescription',
          'Radiology Report', 'Insurance Claim', 'Referral Letter',
          'Clinical Notes', 'Pathology Report',
          'Patient Intake Form', 'Nursing Notes'
        )
      ):labels[0]::VARCHAR                                  AS DOC_CATEGORY,

      NULL::FLOAT                                           AS DOC_CATEGORY_CONFIDENCE,

      -----------------------------------------------------------
      -- 3. AI_SENTIMENT — overall + multi-dimensional
      -----------------------------------------------------------
      SNOWFLAKE.CORTEX.SENTIMENT(t.FILE_TEXT)               AS SENTIMENT_SCORE,

      AI_SENTIMENT(t.FILE_TEXT)                              AS SENTIMENT_DIMENSIONS,

      -----------------------------------------------------------
      -- 4. AI_SUMMARIZE — concise summary
      -----------------------------------------------------------
      SNOWFLAKE.CORTEX.SUMMARIZE(t.FILE_TEXT)               AS SUMMARY,

      -----------------------------------------------------------
      -- 5. AI_TRANSLATE — detect language + translate if needed
      -----------------------------------------------------------
      AI_CLASSIFY(
        t.FILE_TEXT,
        ARRAY_CONSTRUCT('English', 'Spanish', 'French', 'German',
                        'Portuguese', 'Chinese', 'Japanese', 'Korean', 'Arabic')
      ):labels[0]::VARCHAR                                  AS DETECTED_LANGUAGE,

      CASE
        WHEN AI_CLASSIFY(
          t.FILE_TEXT,
          ARRAY_CONSTRUCT('English', 'Spanish', 'French', 'German',
                          'Portuguese', 'Chinese', 'Japanese', 'Korean', 'Arabic')
        ):labels[0]::VARCHAR != 'English'
        THEN AI_TRANSLATE(t.FILE_TEXT, '', 'en')
        ELSE NULL
      END                                                   AS TRANSLATED_TEXT,

      -----------------------------------------------------------
      -- 6. AI_REDACT — remove PII
      -----------------------------------------------------------
      AI_REDACT(t.FILE_TEXT)                                AS REDACTED_TEXT,

      -----------------------------------------------------------
      -- 7. AI_COMPLETE — key insights and action items
      -----------------------------------------------------------
      AI_COMPLETE(
        'claude-3-5-sonnet',
        CONCAT(
          'You are a medical document analyst. Given the following medical document, ',
          'provide: 1) Three key clinical insights, 2) Any urgent action items, ',
          '3) Recommended follow-ups. Be concise. Document: ',
          t.FILE_TEXT
        )
      )                                                     AS KEY_INSIGHTS,

      -----------------------------------------------------------
      -- 8. AI_EMBED — vector embedding for Cortex Search
      -----------------------------------------------------------
      AI_EMBED('snowflake-arctic-embed-m-v1.5', t.FILE_TEXT) AS EMBEDDING

  FROM txt_content t;

  -- Mark TXT files as processed
  UPDATE RAW.FILES_LOG
  SET IS_PROCESSED = TRUE,
      PROCESSED_AT = CURRENT_TIMESTAMP()
  WHERE FILE_TYPE = 'TXT'
    AND IS_PROCESSED = FALSE;

  RETURN 'TXT processing complete: ' || CURRENT_TIMESTAMP()::VARCHAR;

END;
$$

### Audio Processing Procedure

In [ ]:
CREATE OR REPLACE PROCEDURE PROCESSED.PROCESS_AUDIO_FILES()
  RETURNS VARCHAR
  LANGUAGE SQL
  COMMENT = 'Processes unprocessed WAV/MP3 files through 8 Cortex AI functions into AUDIO_INTELLIGENCE'
  EXECUTE AS CALLER
AS
$$
BEGIN

  INSERT INTO PROCESSED.AUDIO_INTELLIGENCE (
      FILE_ID, FILE_NAME, TRANSCRIPT_TEXT, TRANSCRIPT_SEGMENTS,
      AUDIO_DURATION_SECS, SPEAKER_COUNT,
      EXTRACTED_FIELDS, CALL_CATEGORY, CALL_CATEGORY_CONFIDENCE,
      SENTIMENT_SCORE, SENTIMENT_DIMENSIONS, SUMMARY,
      DETECTED_LANGUAGE, TRANSLATED_TRANSCRIPT,
      CONSULTATION_NOTES, EMBEDDING
  )
  WITH transcribed AS (
    SELECT
      f.FILE_ID,
      f.FILE_NAME,
      AI_TRANSCRIBE(
        TO_FILE('@RAW.S3_MEDICAL_AUDIO', f.FILE_NAME)
      ) AS RAW_TRANSCRIPT
    FROM RAW.FILES_LOG f
    WHERE f.FILE_TYPE IN ('WAV', 'MP3')
      AND f.IS_PROCESSED = FALSE
  )
  SELECT
      t.FILE_ID,
      t.FILE_NAME,

      -----------------------------------------------------------
      -- 1. AI_TRANSCRIBE — full text
      -----------------------------------------------------------
      t.RAW_TRANSCRIPT:text::VARCHAR                        AS TRANSCRIPT_TEXT,

      -----------------------------------------------------------
      -- 2. AI_TRANSCRIBE — segments with timestamps
      -----------------------------------------------------------
      t.RAW_TRANSCRIPT:segments                             AS TRANSCRIPT_SEGMENTS,

      t.RAW_TRANSCRIPT:audio_duration::FLOAT                AS AUDIO_DURATION_SECS,

      NULL::INT                                             AS SPEAKER_COUNT,

      -----------------------------------------------------------
      -- 3. AI_EXTRACT — structured fields from transcript
      -----------------------------------------------------------
      AI_EXTRACT(
        t.RAW_TRANSCRIPT:text::VARCHAR,
        OBJECT_CONSTRUCT(
          'patient_name',        'string: full name of the patient',
          'provider_name',       'string: name of the doctor or provider',
          'chief_complaint',     'string: main reason for the consultation',
          'diagnosis_discussed', 'string: any diagnosis or condition discussed',
          'medications_discussed', 'array: medications mentioned',
          'follow_up_actions',   'array: action items or follow-ups agreed upon',
          'next_appointment',    'string: date or timeframe of next visit if mentioned'
        )
      )                                                     AS EXTRACTED_FIELDS,

      -----------------------------------------------------------
      -- 4. AI_CLASSIFY — categorize consultation type
      -----------------------------------------------------------
      AI_CLASSIFY(
        t.RAW_TRANSCRIPT:text::VARCHAR,
        ARRAY_CONSTRUCT(
          'Initial Consultation', 'Follow-Up Visit',
          'Specialist Referral', 'Prescription Review',
          'Lab Results Discussion', 'Mental Health Session',
          'Insurance Discussion', 'Emergency Consultation'
        )
      ):labels[0]::VARCHAR                                  AS CALL_CATEGORY,

      NULL::FLOAT                                           AS CALL_CATEGORY_CONFIDENCE,

      -----------------------------------------------------------
      -- 5. AI_SENTIMENT — overall + multi-dimensional
      -----------------------------------------------------------
      SNOWFLAKE.CORTEX.SENTIMENT(t.RAW_TRANSCRIPT:text::VARCHAR) AS SENTIMENT_SCORE,

      AI_SENTIMENT(t.RAW_TRANSCRIPT:text::VARCHAR)          AS SENTIMENT_DIMENSIONS,

      -----------------------------------------------------------
      -- 6. AI_SUMMARIZE — executive summary
      -----------------------------------------------------------
      SNOWFLAKE.CORTEX.SUMMARIZE(t.RAW_TRANSCRIPT:text::VARCHAR) AS SUMMARY,

      -----------------------------------------------------------
      -- 7. AI_TRANSLATE — detect language + translate if needed
      -----------------------------------------------------------
      AI_CLASSIFY(
        t.RAW_TRANSCRIPT:text::VARCHAR,
        ARRAY_CONSTRUCT('English', 'Spanish', 'French', 'German',
                        'Portuguese', 'Chinese', 'Japanese', 'Korean', 'Arabic')
      ):labels[0]::VARCHAR                                  AS DETECTED_LANGUAGE,

      CASE
        WHEN AI_CLASSIFY(
          t.RAW_TRANSCRIPT:text::VARCHAR,
          ARRAY_CONSTRUCT('English', 'Spanish', 'French', 'German',
                          'Portuguese', 'Chinese', 'Japanese', 'Korean', 'Arabic')
        ):labels[0]::VARCHAR != 'English'
        THEN AI_TRANSLATE(t.RAW_TRANSCRIPT:text::VARCHAR, '', 'en')
        ELSE NULL
      END                                                   AS TRANSLATED_TRANSCRIPT,

      -----------------------------------------------------------
      -- 8. AI_COMPLETE — structured SOAP consultation notes
      -----------------------------------------------------------
      AI_COMPLETE(
        'claude-3-5-sonnet',
        CONCAT(
          'You are a medical scribe. Given the following patient consultation transcript, ',
          'generate structured SOAP notes (Subjective, Objective, Assessment, Plan). ',
          'Be concise and clinically accurate. Transcript: ',
          t.RAW_TRANSCRIPT:text::VARCHAR
        )
      )                                                     AS CONSULTATION_NOTES,

      -----------------------------------------------------------
      -- 9. AI_EMBED — vector embedding for Cortex Search
      -----------------------------------------------------------
      AI_EMBED(
        'snowflake-arctic-embed-m-v1.5',
        t.RAW_TRANSCRIPT:text::VARCHAR
      )                                                     AS EMBEDDING

  FROM transcribed t;

  -- Mark audio files as processed
  UPDATE RAW.FILES_LOG
  SET IS_PROCESSED = TRUE,
      PROCESSED_AT = CURRENT_TIMESTAMP()
  WHERE FILE_TYPE IN ('WAV', 'MP3')
    AND IS_PROCESSED = FALSE;

  RETURN 'Audio processing complete: ' || CURRENT_TIMESTAMP()::VARCHAR;

END;
$$

### Run Processing (5-10 minutes total)

In [ ]:
CALL HEALTHCARE_AI_DEMO.PROCESSED.PROCESS_PDF_FILES();

In [ ]:
CALL HEALTHCARE_AI_DEMO.PROCESSED.PROCESS_TXT_FILES();

In [ ]:
CALL HEALTHCARE_AI_DEMO.PROCESSED.PROCESS_AUDIO_FILES();

### Verify (expected: 6 PDF, 4 TXT, 5 Audio)

In [ ]:
SELECT 'PDF' AS TYPE, COUNT(*) AS CNT FROM HEALTHCARE_AI_DEMO.PROCESSED.PDF_INTELLIGENCE
UNION ALL SELECT 'TXT', COUNT(*) FROM HEALTHCARE_AI_DEMO.PROCESSED.TXT_INTELLIGENCE
UNION ALL SELECT 'AUDIO', COUNT(*) FROM HEALTHCARE_AI_DEMO.PROCESSED.AUDIO_INTELLIGENCE;

## Step 7: Cortex Search Services

Semantic retrieval over AI-enriched content. One service per file type with filterable attributes.

> Wait 2-3 minutes after creation for indexing.

In [ ]:
-----------------------------------------------------------------------
-- 1. PDF SEARCH — over parsed & enriched PDF content
-----------------------------------------------------------------------
CREATE OR REPLACE CORTEX SEARCH SERVICE PROCESSED.PDF_SEARCH
  ON SEARCH_TEXT
  ATTRIBUTES DOC_CATEGORY, PATIENT_NAME, PROVIDER_NAME, DIAGNOSIS, DETECTED_LANGUAGE, SENTIMENT_SCORE, PROCESSED_AT
  WAREHOUSE = HEALTHCARE_AI_WH
  TARGET_LAG = '1 hour'
  EMBEDDING_MODEL = 'snowflake-arctic-embed-m-v1.5'
  COMMENT = 'Cortex Search over AI-parsed medical PDF documents'
AS (
    SELECT
      DOC_ID,
      FILE_NAME,
      CONCAT(
        COALESCE(PARSED_TEXT, ''), '\n\n',
        'SUMMARY: ', COALESCE(SUMMARY, ''), '\n\n',
        'INSIGHTS: ', COALESCE(KEY_INSIGHTS, ''), '\n\n',
        'DIAGNOSIS: ', COALESCE(EXTRACTED_FIELDS:diagnosis::VARCHAR, '')
      )                                                   AS SEARCH_TEXT,
      DOC_CATEGORY,
      COALESCE(EXTRACTED_FIELDS:patient_name::VARCHAR, 'Unknown')  AS PATIENT_NAME,
      COALESCE(EXTRACTED_FIELDS:provider_name::VARCHAR, 'Unknown') AS PROVIDER_NAME,
      COALESCE(EXTRACTED_FIELDS:diagnosis::VARCHAR, '')   AS DIAGNOSIS,
      DETECTED_LANGUAGE,
      SENTIMENT_SCORE::VARCHAR                            AS SENTIMENT_SCORE,
      PROCESSED_AT::VARCHAR                               AS PROCESSED_AT
    FROM PROCESSED.PDF_INTELLIGENCE
);

-----------------------------------------------------------------------
-- 2. TXT SEARCH — over enriched text document content
-----------------------------------------------------------------------
CREATE OR REPLACE CORTEX SEARCH SERVICE PROCESSED.TXT_SEARCH
  ON SEARCH_TEXT
  ATTRIBUTES DOC_CATEGORY, PATIENT_NAME, PROVIDER_NAME, DIAGNOSIS, DETECTED_LANGUAGE, SENTIMENT_SCORE, PROCESSED_AT
  WAREHOUSE = HEALTHCARE_AI_WH
  TARGET_LAG = '1 hour'
  EMBEDDING_MODEL = 'snowflake-arctic-embed-m-v1.5'
  COMMENT = 'Cortex Search over AI-enriched medical text documents'
AS (
    SELECT
      TXT_ID,
      FILE_NAME,
      CONCAT(
        COALESCE(RAW_TEXT, ''), '\n\n',
        'SUMMARY: ', COALESCE(SUMMARY, ''), '\n\n',
        'INSIGHTS: ', COALESCE(KEY_INSIGHTS, ''), '\n\n',
        'DIAGNOSIS: ', COALESCE(EXTRACTED_FIELDS:diagnosis::VARCHAR, '')
      )                                                   AS SEARCH_TEXT,
      DOC_CATEGORY,
      COALESCE(EXTRACTED_FIELDS:patient_name::VARCHAR, 'Unknown')  AS PATIENT_NAME,
      COALESCE(EXTRACTED_FIELDS:provider_name::VARCHAR, 'Unknown') AS PROVIDER_NAME,
      COALESCE(EXTRACTED_FIELDS:diagnosis::VARCHAR, '')   AS DIAGNOSIS,
      DETECTED_LANGUAGE,
      SENTIMENT_SCORE::VARCHAR                            AS SENTIMENT_SCORE,
      PROCESSED_AT::VARCHAR                               AS PROCESSED_AT
    FROM PROCESSED.TXT_INTELLIGENCE
);

-----------------------------------------------------------------------
-- 3. AUDIO SEARCH — over transcribed consultation recordings
-----------------------------------------------------------------------
CREATE OR REPLACE CORTEX SEARCH SERVICE PROCESSED.AUDIO_SEARCH
  ON SEARCH_TEXT
  ATTRIBUTES CALL_CATEGORY, PATIENT_NAME, PROVIDER_NAME, CHIEF_COMPLAINT, DURATION_SECS, SPEAKER_COUNT, DETECTED_LANGUAGE, SENTIMENT_SCORE, PROCESSED_AT
  WAREHOUSE = HEALTHCARE_AI_WH
  TARGET_LAG = '1 hour'
  EMBEDDING_MODEL = 'snowflake-arctic-embed-m-v1.5'
  COMMENT = 'Cortex Search over transcribed patient consultations (WAV/MP3 audio)'
AS (
    SELECT
      AUDIO_ID,
      FILE_NAME,
      CONCAT(
        COALESCE(TRANSCRIPT_TEXT, ''), '\n\n',
        'SUMMARY: ', COALESCE(SUMMARY, ''), '\n\n',
        'CONSULTATION NOTES: ', COALESCE(CONSULTATION_NOTES, ''), '\n\n',
        'CHIEF COMPLAINT: ', COALESCE(EXTRACTED_FIELDS:chief_complaint::VARCHAR, '')
      )                                                   AS SEARCH_TEXT,
      CALL_CATEGORY,
      COALESCE(EXTRACTED_FIELDS:patient_name::VARCHAR, 'Unknown')  AS PATIENT_NAME,
      COALESCE(EXTRACTED_FIELDS:provider_name::VARCHAR, 'Unknown') AS PROVIDER_NAME,
      COALESCE(EXTRACTED_FIELDS:chief_complaint::VARCHAR, '')      AS CHIEF_COMPLAINT,
      AUDIO_DURATION_SECS::VARCHAR                        AS DURATION_SECS,
      SPEAKER_COUNT::VARCHAR                              AS SPEAKER_COUNT,
      DETECTED_LANGUAGE,
      SENTIMENT_SCORE::VARCHAR                            AS SENTIMENT_SCORE,
      PROCESSED_AT::VARCHAR                               AS PROCESSED_AT
    FROM PROCESSED.AUDIO_INTELLIGENCE
);

-----------------------------------------------------------------------
-- 4. VERIFY
-----------------------------------------------------------------------
SHOW CORTEX SEARCH SERVICES IN DATABASE HEALTHCARE_AI_DEMO;

In [ ]:
SHOW CORTEX SEARCH SERVICES IN DATABASE HEALTHCARE_AI_DEMO;

## Step 8: Structured Data

Load sample healthcare operational data: Providers (12), Patients (15), Claims (30), Appointments (24).

In [ ]:
-----------------------------------------------------------------------
-- 1. PROVIDERS — doctors, specialists, facilities
-----------------------------------------------------------------------
CREATE OR REPLACE TABLE ANALYTICS.PROVIDERS (
    PROVIDER_ID     NUMBER PRIMARY KEY,
    PROVIDER_NAME   VARCHAR        NOT NULL,
    SPECIALTY       VARCHAR(100)   NOT NULL,
    FACILITY_NAME   VARCHAR(200),
    CITY            VARCHAR(100),
    STATE           VARCHAR(50),
    NPI_NUMBER      VARCHAR(20)    COMMENT 'National Provider Identifier',
    IS_ACTIVE       BOOLEAN        DEFAULT TRUE
);

INSERT INTO ANALYTICS.PROVIDERS VALUES
  (1,  'Dr. Sarah Chen',       'Internal Medicine',    'Mercy General Hospital',        'Hartford',      'CT', '1234567890', TRUE),
  (2,  'Dr. James Okafor',     'Cardiology',           'Mercy General Hospital',        'Hartford',      'CT', '1234567891', TRUE),
  (3,  'Dr. Maria Rodriguez',  'Orthopedics',          'St. Luke''s Medical Center',    'New Haven',     'CT', '1234567892', TRUE),
  (4,  'Dr. Robert Kim',       'Neurology',            'Yale New Haven Hospital',       'New Haven',     'CT', '1234567893', TRUE),
  (5,  'Dr. Aisha Patel',      'Oncology',             'Hartford Healthcare Cancer',    'Hartford',      'CT', '1234567894', TRUE),
  (6,  'Dr. David Thompson',   'Psychiatry',           'Silver Hill Hospital',          'New Canaan',    'CT', '1234567895', TRUE),
  (7,  'Dr. Lisa Wang',        'Pediatrics',           'Connecticut Children''s',       'Hartford',      'CT', '1234567896', TRUE),
  (8,  'Dr. Michael Johnson',  'Dermatology',          'Dermatology Associates',        'Stamford',      'CT', '1234567897', TRUE),
  (9,  'Dr. Emily Foster',     'Radiology',            'Mercy General Hospital',        'Hartford',      'CT', '1234567898', TRUE),
  (10, 'Dr. Ahmed Hassan',     'Pulmonology',          'Bridgeport Hospital',           'Bridgeport',    'CT', '1234567899', TRUE),
  (11, 'Dr. Priya Sharma',     'Endocrinology',        'Yale New Haven Hospital',       'New Haven',     'CT', '1234567900', TRUE),
  (12, 'Dr. Carlos Mendez',    'Gastroenterology',     'St. Luke''s Medical Center',    'New Haven',     'CT', '1234567901', TRUE);

-----------------------------------------------------------------------
-- 2. PATIENTS
-----------------------------------------------------------------------
CREATE OR REPLACE TABLE ANALYTICS.PATIENTS (
    PATIENT_ID      NUMBER PRIMARY KEY,
    FIRST_NAME      VARCHAR(100)   NOT NULL,
    LAST_NAME       VARCHAR(100)   NOT NULL,
    DATE_OF_BIRTH   DATE           NOT NULL,
    GENDER          VARCHAR(20),
    CITY            VARCHAR(100),
    STATE           VARCHAR(50),
    ZIP_CODE        VARCHAR(10),
    INSURANCE_PLAN  VARCHAR(200),
    PRIMARY_PROVIDER_ID NUMBER     COMMENT 'FK to PROVIDERS',
    REGISTERED_DATE DATE           DEFAULT CURRENT_DATE()
);

INSERT INTO ANALYTICS.PATIENTS VALUES
  (101, 'John',     'Whitfield',  '1958-03-15', 'Male',   'Hartford',     'CT', '06101', 'Aetna PPO',                1,  '2023-01-10'),
  (102, 'Margaret', 'Sullivan',   '1965-11-22', 'Female', 'New Haven',    'CT', '06510', 'UnitedHealth Choice Plus', 2,  '2023-02-14'),
  (103, 'David',    'Park',       '1972-07-08', 'Male',   'Stamford',     'CT', '06901', 'Cigna Open Access',        3,  '2023-03-20'),
  (104, 'Sandra',   'Williams',   '1980-01-30', 'Female', 'Bridgeport',   'CT', '06601', 'Anthem Blue Cross',        1,  '2023-04-05'),
  (105, 'Roberto',  'Garcia',     '1955-09-12', 'Male',   'Hartford',     'CT', '06103', 'Medicare Advantage',       2,  '2023-05-18'),
  (106, 'Linda',    'Brown',      '1948-12-05', 'Female', 'New Canaan',   'CT', '06840', 'Medicare Advantage',       5,  '2023-06-01'),
  (107, 'Wei',      'Zhang',      '1990-04-17', 'Male',   'New Haven',    'CT', '06511', 'Aetna PPO',                4,  '2023-07-12'),
  (108, 'Patricia', 'O''Brien',   '1975-08-25', 'Female', 'Hartford',     'CT', '06105', 'UnitedHealth Choice Plus', 6,  '2023-08-03'),
  (109, 'Michael',  'Torres',     '1988-02-14', 'Male',   'Bridgeport',   'CT', '06604', 'Cigna Open Access',        10, '2023-09-15'),
  (110, 'Amara',    'Johnson',    '2015-06-20', 'Female', 'Hartford',     'CT', '06106', 'Anthem Blue Cross',        7,  '2023-10-22'),
  (111, 'Catherine','Lee',        '1960-10-03', 'Female', 'Stamford',     'CT', '06902', 'Aetna PPO',                11, '2023-11-08'),
  (112, 'James',    'Murphy',     '1970-05-19', 'Male',   'New Haven',    'CT', '06512', 'Medicare Advantage',       12, '2024-01-05'),
  (113, 'Fatima',   'Ali',        '1985-03-28', 'Female', 'Hartford',     'CT', '06102', 'UnitedHealth Choice Plus', 1,  '2024-02-14'),
  (114, 'Thomas',   'Anderson',   '1952-07-11', 'Male',   'New Canaan',   'CT', '06840', 'Medicare Advantage',       2,  '2024-03-01'),
  (115, 'Yuki',     'Tanaka',     '1995-01-25', 'Female', 'New Haven',    'CT', '06513', 'Cigna Open Access',        8,  '2024-04-10');

-----------------------------------------------------------------------
-- 3. CLAIMS
-----------------------------------------------------------------------
CREATE OR REPLACE TABLE ANALYTICS.CLAIMS (
    CLAIM_ID         NUMBER PRIMARY KEY,
    PATIENT_ID       NUMBER         NOT NULL COMMENT 'FK to PATIENTS',
    PROVIDER_ID      NUMBER         NOT NULL COMMENT 'FK to PROVIDERS',
    SERVICE_DATE     DATE           NOT NULL,
    CLAIM_DATE       DATE           NOT NULL,
    PROCEDURE_CODE   VARCHAR(20)    COMMENT 'CPT code',
    PROCEDURE_DESC   VARCHAR(500),
    DIAGNOSIS_CODE   VARCHAR(20)    COMMENT 'ICD-10 code',
    DIAGNOSIS_DESC   VARCHAR(500),
    BILLED_AMOUNT    NUMBER(10,2),
    ALLOWED_AMOUNT   NUMBER(10,2),
    PAID_AMOUNT      NUMBER(10,2),
    PATIENT_RESPONSIBILITY NUMBER(10,2),
    CLAIM_STATUS     VARCHAR(50)    COMMENT 'Approved, Denied, Pending, Under Review',
    DENIAL_REASON    VARCHAR(500)
);

INSERT INTO ANALYTICS.CLAIMS VALUES
  (1001, 101, 1, '2024-06-10', '2024-06-12', '99213', 'Office visit - established patient (moderate)',     'I10',    'Essential hypertension',                     185.00, 142.00, 113.60, 28.40, 'Approved', NULL),
  (1002, 101, 2, '2024-06-15', '2024-06-17', '93000', 'Electrocardiogram (ECG)',                           'I10',    'Essential hypertension',                     350.00, 280.00, 224.00, 56.00, 'Approved', NULL),
  (1003, 102, 2, '2024-07-01', '2024-07-03', '93306', 'Echocardiography, transthoracic',                   'I25.10', 'Atherosclerotic heart disease',              1200.00, 960.00, 768.00, 192.00, 'Approved', NULL),
  (1004, 103, 3, '2024-07-10', '2024-07-12', '27447', 'Total knee replacement',                            'M17.11', 'Primary osteoarthritis, right knee',         45000.00, 38000.00, 30400.00, 7600.00, 'Approved', NULL),
  (1005, 104, 1, '2024-07-20', '2024-07-22', '99214', 'Office visit - established patient (moderate-high)', 'J06.9',  'Acute upper respiratory infection',          250.00, 195.00, 156.00, 39.00, 'Approved', NULL),
  (1006, 105, 2, '2024-08-05', '2024-08-07', '93458', 'Left heart catheterization',                        'I25.10', 'Atherosclerotic heart disease',              8500.00, 7200.00, 5760.00, 1440.00, 'Approved', NULL),
  (1007, 106, 5, '2024-08-12', '2024-08-14', '88305', 'Surgical pathology - tissue exam',                   'C50.911','Malignant neoplasm, right breast',           425.00, 340.00, 272.00, 68.00, 'Approved', NULL),
  (1008, 107, 4, '2024-08-20', '2024-08-22', '95819', 'Electroencephalogram (EEG) with sleep',             'G40.909','Epilepsy, unspecified',                       800.00, 640.00, 512.00, 128.00, 'Approved', NULL),
  (1009, 108, 6, '2024-09-01', '2024-09-03', '90834', 'Psychotherapy, 45 minutes',                         'F33.0',  'Major depressive disorder, recurrent, mild', 200.00, 180.00, 144.00, 36.00, 'Approved', NULL),
  (1010, 109, 10,'2024-09-10', '2024-09-12', '94010', 'Spirometry (pulmonary function test)',               'J45.20', 'Mild intermittent asthma',                   275.00, 220.00, 176.00, 44.00, 'Approved', NULL),
  (1011, 110, 7, '2024-09-18', '2024-09-20', '99392', 'Preventive visit - age 1-4',                        'Z00.129','Routine child health exam',                  225.00, 200.00, 160.00, 40.00, 'Approved', NULL),
  (1012, 111, 11,'2024-10-01', '2024-10-03', '83036', 'Hemoglobin A1c test',                               'E11.65', 'Type 2 diabetes with hyperglycemia',         85.00,  68.00,  54.40,  13.60, 'Approved', NULL),
  (1013, 112, 12,'2024-10-15', '2024-10-17', '43239', 'Upper GI endoscopy with biopsy',                    'K21.0',  'GERD with esophagitis',                      3200.00, 2700.00, 2160.00, 540.00, 'Approved', NULL),
  (1014, 113, 1, '2024-10-22', '2024-10-24', '99395', 'Preventive visit - age 18-39',                      'Z00.00', 'General adult medical exam',                 275.00, 225.00, 180.00, 45.00, 'Approved', NULL),
  (1015, 114, 2, '2024-11-01', '2024-11-03', '33533', 'Coronary artery bypass graft (CABG)',                'I25.10', 'Atherosclerotic heart disease',              85000.00, 72000.00, 57600.00, 14400.00, 'Approved', NULL),
  (1016, 101, 1, '2024-11-10', '2024-11-12', '99214', 'Office visit - established patient (moderate-high)', 'I10',    'Essential hypertension',                     250.00, 195.00, 156.00, 39.00, 'Approved', NULL),
  (1017, 102, 2, '2024-11-20', '2024-11-22', '93000', 'Electrocardiogram (ECG)',                            'I25.10', 'Atherosclerotic heart disease',              350.00, 280.00, 224.00, 56.00, 'Approved', NULL),
  (1018, 105, 2, '2024-12-01', '2024-12-03', '99215', 'Office visit - established patient (high)',          'I25.10', 'Atherosclerotic heart disease',              375.00, 300.00, 240.00, 60.00, 'Approved', NULL),
  (1019, 106, 5, '2024-12-10', '2024-12-12', '77063', 'Screening mammography, bilateral',                  'Z12.31', 'Screening mammogram',                        285.00, 250.00, 200.00, 50.00, 'Approved', NULL),
  (1020, 108, 6, '2024-12-15', '2024-12-17', '90834', 'Psychotherapy, 45 minutes',                         'F33.1',  'Major depressive disorder, recurrent, moderate', 200.00, 180.00, 144.00, 36.00, 'Pending', NULL),
  (1021, 103, 3, '2025-01-08', '2025-01-10', '99213', 'Office visit - post-op follow-up',                   'M17.11', 'Primary osteoarthritis, right knee',        185.00, 142.00, 113.60, 28.40, 'Approved', NULL),
  (1022, 107, 4, '2025-01-15', '2025-01-17', '70553', 'MRI brain with and without contrast',                'G40.909','Epilepsy, unspecified',                      2800.00, 2200.00, 1760.00, 440.00, 'Approved', NULL),
  (1023, 109, 10,'2025-01-25', '2025-01-27', '99214', 'Office visit - established patient (moderate-high)', 'J45.20', 'Mild intermittent asthma',                  250.00, 195.00, 156.00, 39.00, 'Approved', NULL),
  (1024, 115, 8, '2025-02-01', '2025-02-03', '11102', 'Tangential biopsy of skin',                         'L82.1',  'Seborrheic keratosis',                       350.00, 280.00, 224.00, 56.00, 'Approved', NULL),
  (1025, 113, 9, '2025-02-10', '2025-02-12', '71046', 'Chest X-ray, 2 views',                              'R05.9',  'Cough, unspecified',                         180.00, 145.00, 116.00, 29.00, 'Denied', 'Prior authorization not obtained'),
  (1026, 111, 11,'2025-02-20', '2025-02-22', '80053', 'Comprehensive metabolic panel',                      'E11.65', 'Type 2 diabetes with hyperglycemia',        125.00, 100.00, 80.00,  20.00, 'Approved', NULL),
  (1027, 112, 12,'2025-03-01', '2025-03-03', '99214', 'Office visit - GI follow-up',                        'K21.0',  'GERD with esophagitis',                     250.00, 195.00, 156.00, 39.00, 'Approved', NULL),
  (1028, 104, 1, '2025-03-10', '2025-03-12', '36415', 'Venipuncture (blood draw)',                          'Z00.00', 'General adult medical exam',                 35.00,  28.00,  22.40,  5.60,  'Approved', NULL),
  (1029, 110, 7, '2025-03-15', '2025-03-17', '90460', 'Immunization administration',                       'Z23',    'Immunization encounter',                     65.00,  55.00,  44.00,  11.00, 'Approved', NULL),
  (1030, 114, 2, '2025-03-20', '2025-03-22', '99215', 'Office visit - post-CABG follow-up',                 'I25.10', 'Atherosclerotic heart disease',             375.00, 300.00, 240.00, 60.00, 'Under Review', NULL);

-----------------------------------------------------------------------
-- 4. APPOINTMENTS
-----------------------------------------------------------------------
CREATE OR REPLACE TABLE ANALYTICS.APPOINTMENTS (
    APPOINTMENT_ID   NUMBER PRIMARY KEY,
    PATIENT_ID       NUMBER         NOT NULL,
    PROVIDER_ID      NUMBER         NOT NULL,
    APPOINTMENT_DATE TIMESTAMP_NTZ  NOT NULL,
    APPOINTMENT_TYPE VARCHAR(100)   COMMENT 'In-Person, Telehealth, Phone',
    VISIT_REASON     VARCHAR(500),
    DURATION_MINUTES NUMBER,
    STATUS           VARCHAR(50)    COMMENT 'Completed, Cancelled, No-Show, Scheduled',
    HAS_AUDIO_RECORDING BOOLEAN    DEFAULT FALSE COMMENT 'Whether consultation was recorded',
    HAS_DOCUMENT     BOOLEAN       DEFAULT FALSE COMMENT 'Whether a document was generated'
);

INSERT INTO ANALYTICS.APPOINTMENTS VALUES
  (2001, 101, 1, '2024-06-10 09:00:00', 'In-Person',  'Hypertension follow-up, medication review',         30, 'Completed',  TRUE,  TRUE),
  (2002, 101, 2, '2024-06-15 10:30:00', 'In-Person',  'Cardiology referral — ECG and evaluation',          45, 'Completed',  TRUE,  TRUE),
  (2003, 102, 2, '2024-07-01 08:00:00', 'In-Person',  'Echocardiogram and cardiac assessment',             60, 'Completed',  FALSE, TRUE),
  (2004, 103, 3, '2024-07-10 07:00:00', 'In-Person',  'Total knee replacement surgery',                    180,'Completed',  FALSE, TRUE),
  (2005, 104, 1, '2024-07-20 14:00:00', 'Telehealth', 'Acute respiratory infection — virtual visit',       20, 'Completed',  TRUE,  TRUE),
  (2006, 105, 2, '2024-08-05 09:00:00', 'In-Person',  'Cardiac catheterization, pre-op assessment',        90, 'Completed',  TRUE,  TRUE),
  (2007, 106, 5, '2024-08-12 11:00:00', 'In-Person',  'Pathology results review — breast biopsy',          30, 'Completed',  TRUE,  TRUE),
  (2008, 107, 4, '2024-08-20 13:00:00', 'In-Person',  'EEG with sleep study — seizure evaluation',         120,'Completed',  FALSE, TRUE),
  (2009, 108, 6, '2024-09-01 15:00:00', 'Telehealth', 'Psychotherapy session — depression management',     45, 'Completed',  TRUE,  FALSE),
  (2010, 109, 10,'2024-09-10 10:00:00', 'In-Person',  'Pulmonary function test — asthma evaluation',       40, 'Completed',  TRUE,  TRUE),
  (2011, 110, 7, '2024-09-18 09:30:00', 'In-Person',  'Well-child check-up, immunizations',                30, 'Completed',  FALSE, TRUE),
  (2012, 111, 11,'2024-10-01 08:30:00', 'In-Person',  'Diabetes management — A1c review',                  30, 'Completed',  TRUE,  TRUE),
  (2013, 112, 12,'2024-10-15 07:30:00', 'In-Person',  'Upper GI endoscopy — GERD evaluation',              60, 'Completed',  FALSE, TRUE),
  (2014, 113, 1, '2024-10-22 11:00:00', 'In-Person',  'Annual physical exam',                              45, 'Completed',  TRUE,  TRUE),
  (2015, 114, 2, '2024-11-01 06:00:00', 'In-Person',  'Coronary artery bypass surgery',                    300,'Completed',  FALSE, TRUE),
  (2016, 101, 1, '2024-11-10 09:00:00', 'Telehealth', 'BP check and medication adjustment',                20, 'Completed',  TRUE,  TRUE),
  (2017, 108, 6, '2024-12-15 15:00:00', 'Telehealth', 'Psychotherapy session — progress review',           45, 'Completed',  TRUE,  FALSE),
  (2018, 105, 2, '2024-12-01 10:00:00', 'In-Person',  'Post-catheterization follow-up',                    30, 'Completed',  TRUE,  TRUE),
  (2019, 115, 8, '2025-02-01 14:30:00', 'In-Person',  'Skin lesion evaluation and biopsy',                 25, 'Completed',  FALSE, TRUE),
  (2020, 114, 2, '2025-03-20 09:00:00', 'In-Person',  'Post-CABG cardiac rehabilitation follow-up',        45, 'Completed',  TRUE,  TRUE),
  (2021, 103, 3, '2025-04-15 10:00:00', 'In-Person',  'Post-op knee rehab assessment',                     30, 'Scheduled',  FALSE, FALSE),
  (2022, 107, 4, '2025-04-20 13:00:00', 'Telehealth', 'Epilepsy medication review',                        30, 'Scheduled',  FALSE, FALSE),
  (2023, 106, 5, '2025-04-25 11:00:00', 'In-Person',  'Oncology follow-up — 6-month imaging',              45, 'Scheduled',  FALSE, FALSE),
  (2024, 109, 10,'2025-05-01 10:00:00', 'Phone',      'Asthma action plan review',                         15, 'Scheduled',  FALSE, FALSE);

-----------------------------------------------------------------------
-- 5. VERIFY ROW COUNTS
-----------------------------------------------------------------------
SELECT 'PROVIDERS' AS TBL, COUNT(*) AS "ROWS" FROM ANALYTICS.PROVIDERS
UNION ALL SELECT 'PATIENTS', COUNT(*) FROM ANALYTICS.PATIENTS
UNION ALL SELECT 'CLAIMS', COUNT(*) FROM ANALYTICS.CLAIMS
UNION ALL SELECT 'APPOINTMENTS', COUNT(*) FROM ANALYTICS.APPOINTMENTS;

## Step 9: Semantic View for Cortex Analyst

Defines tables, relationships, dimensions, and metrics for natural-language-to-SQL.

In [ ]:
-----------------------------------------------------------------------
-- SEMANTIC VIEW
-----------------------------------------------------------------------
CREATE OR REPLACE SEMANTIC VIEW ANALYTICS.HEALTHCARE_ANALYTICS

  TABLES (
    PATIENTS AS ANALYTICS.PATIENTS
      PRIMARY KEY (PATIENT_ID)
      COMMENT = 'Patient demographics and registration',

    PROVIDERS AS ANALYTICS.PROVIDERS
      PRIMARY KEY (PROVIDER_ID)
      COMMENT = 'Provider directory and specialties',

    CLAIMS AS ANALYTICS.CLAIMS
      PRIMARY KEY (CLAIM_ID)
      COMMENT = 'Claims data including billing, diagnosis, procedures, and status',

    APPOINTMENTS AS ANALYTICS.APPOINTMENTS
      PRIMARY KEY (APPOINTMENT_ID)
      COMMENT = 'Appointment scheduling, visit types, and recording status'
  )

  RELATIONSHIPS (
    patients_to_providers AS
      PATIENTS (PRIMARY_PROVIDER_ID) REFERENCES PROVIDERS (PROVIDER_ID),
    claims_to_patients AS
      CLAIMS (PATIENT_ID) REFERENCES PATIENTS (PATIENT_ID),
    claims_to_providers AS
      CLAIMS (PROVIDER_ID) REFERENCES PROVIDERS (PROVIDER_ID),
    appointments_to_patients AS
      APPOINTMENTS (PATIENT_ID) REFERENCES PATIENTS (PATIENT_ID),
    appointments_to_providers AS
      APPOINTMENTS (PROVIDER_ID) REFERENCES PROVIDERS (PROVIDER_ID)
  )

  DIMENSIONS (
    PATIENTS.first_name AS FIRST_NAME
      COMMENT = 'Patient first name',
    PATIENTS.last_name AS LAST_NAME
      COMMENT = 'Patient last name',
    PATIENTS.date_of_birth AS DATE_OF_BIRTH
      COMMENT = 'Patient date of birth',
    PATIENTS.gender AS GENDER
      COMMENT = 'Patient gender (Male, Female)',
    PATIENTS.city AS CITY
      COMMENT = 'Patient city of residence',
    PATIENTS.state AS STATE
      COMMENT = 'Patient state of residence',
    PATIENTS.zip_code AS ZIP_CODE
      COMMENT = 'Patient zip code',
    PATIENTS.insurance_plan AS INSURANCE_PLAN
      COMMENT = 'Insurance plan name (Aetna PPO, UnitedHealth Choice Plus, Cigna Open Access, Anthem Blue Cross, Medicare Advantage)',
    PATIENTS.registered_date AS REGISTERED_DATE
      COMMENT = 'Date patient was registered in the system',

    PROVIDERS.provider_name AS PROVIDER_NAME
      COMMENT = 'Full name of the doctor or provider',
    PROVIDERS.specialty AS SPECIALTY
      COMMENT = 'Medical specialty (Internal Medicine, Cardiology, Orthopedics, Neurology, Oncology, Psychiatry, Pediatrics, Dermatology, Radiology, Pulmonology, Endocrinology, Gastroenterology)',
    PROVIDERS.facility_name AS FACILITY_NAME
      COMMENT = 'Hospital or clinic name where provider practices',
    PROVIDERS.provider_city AS PROVIDERS.CITY
      COMMENT = 'City where the facility is located',
    PROVIDERS.provider_state AS PROVIDERS.STATE
      COMMENT = 'State where the facility is located',
    PROVIDERS.npi_number AS NPI_NUMBER
      COMMENT = 'National Provider Identifier number',
    PROVIDERS.is_active AS IS_ACTIVE
      COMMENT = 'Whether the provider is currently active',

    CLAIMS.service_date AS SERVICE_DATE
      COMMENT = 'Date the medical service was performed',
    CLAIMS.claim_date AS CLAIM_DATE
      COMMENT = 'Date the claim was submitted',
    CLAIMS.procedure_code AS PROCEDURE_CODE
      COMMENT = 'CPT procedure code',
    CLAIMS.procedure_desc AS PROCEDURE_DESC
      COMMENT = 'Description of the medical procedure',
    CLAIMS.diagnosis_code AS DIAGNOSIS_CODE
      COMMENT = 'ICD-10 diagnosis code',
    CLAIMS.diagnosis_desc AS DIAGNOSIS_DESC
      COMMENT = 'Description of the diagnosis',
    CLAIMS.claim_status AS CLAIM_STATUS
      COMMENT = 'Status of the claim: Approved, Denied, Pending, Under Review',
    CLAIMS.denial_reason AS DENIAL_REASON
      COMMENT = 'Reason for claim denial if applicable',

    APPOINTMENTS.appointment_date AS APPOINTMENT_DATE
      COMMENT = 'Date and time of the appointment',
    APPOINTMENTS.appointment_type AS APPOINTMENT_TYPE
      COMMENT = 'Type of visit: In-Person, Telehealth, Phone',
    APPOINTMENTS.visit_reason AS VISIT_REASON
      COMMENT = 'Reason for the visit or chief complaint',
    APPOINTMENTS.status AS STATUS
      COMMENT = 'Appointment status: Completed, Cancelled, No-Show, Scheduled',
    APPOINTMENTS.has_audio_recording AS HAS_AUDIO_RECORDING
      COMMENT = 'Whether the consultation was recorded as audio (TRUE/FALSE)',
    APPOINTMENTS.has_document AS HAS_DOCUMENT
      COMMENT = 'Whether a medical document was generated (TRUE/FALSE)'
  )

  METRICS (
    CLAIMS.total_billed_amount AS SUM(BILLED_AMOUNT)
      COMMENT = 'Total amount billed by the provider in dollars',
    CLAIMS.avg_billed_amount AS AVG(BILLED_AMOUNT)
      COMMENT = 'Average amount billed by the provider in dollars',
    CLAIMS.total_allowed_amount AS SUM(ALLOWED_AMOUNT)
      COMMENT = 'Total amount allowed by the insurance plan in dollars',
    CLAIMS.avg_allowed_amount AS AVG(ALLOWED_AMOUNT)
      COMMENT = 'Average amount allowed by the insurance plan in dollars',
    CLAIMS.total_paid_amount AS SUM(PAID_AMOUNT)
      COMMENT = 'Total amount paid by insurance in dollars',
    CLAIMS.avg_paid_amount AS AVG(PAID_AMOUNT)
      COMMENT = 'Average amount paid by insurance in dollars',
    CLAIMS.total_patient_responsibility AS SUM(PATIENT_RESPONSIBILITY)
      COMMENT = 'Total amount the patient owes (copay, coinsurance, deductible) in dollars',
    CLAIMS.avg_patient_responsibility AS AVG(PATIENT_RESPONSIBILITY)
      COMMENT = 'Average amount the patient owes in dollars',
    CLAIMS.claim_count AS COUNT(CLAIM_ID)
      COMMENT = 'Number of claims',

    APPOINTMENTS.total_duration_minutes AS SUM(DURATION_MINUTES)
      COMMENT = 'Total duration of appointments in minutes',
    APPOINTMENTS.avg_duration_minutes AS AVG(DURATION_MINUTES)
      COMMENT = 'Average duration of appointments in minutes',
    APPOINTMENTS.appointment_count AS COUNT(APPOINTMENT_ID)
      COMMENT = 'Number of appointments',

    PATIENTS.patient_count AS COUNT(PATIENT_ID)
      COMMENT = 'Number of patients',

    PROVIDERS.provider_count AS COUNT(PROVIDER_ID)
      COMMENT = 'Number of providers'
  )

  COMMENT = 'Healthcare analytics semantic view for Cortex Analyst. Covers patients, providers, claims, and appointments.';

-----------------------------------------------------------------------
-- VERIFY
-----------------------------------------------------------------------
SHOW SEMANTIC VIEWS IN SCHEMA ANALYTICS;
DESCRIBE SEMANTIC VIEW ANALYTICS.HEALTHCARE_ANALYTICS;

In [ ]:
SHOW SEMANTIC VIEWS IN SCHEMA HEALTHCARE_AI_DEMO.ANALYTICS;

## Step 10: Cortex Agent

Combines 4 tools: HealthcareAnalyst (structured SQL), PDFSearch, TXTSearch, AudioSearch.

In [ ]:
-----------------------------------------------------------------------
-- CORTEX AGENT
-----------------------------------------------------------------------
CREATE OR REPLACE AGENT ANALYTICS.HEALTHCARE_INTELLIGENCE_AGENT
  COMMENT = 'Healthcare intelligence agent combining structured data queries with unstructured search across PDF documents, text documents, and audio consultation transcripts.'
  FROM SPECIFICATION
$$
models:
  orchestration: auto

instructions:
  system: >
    You are a healthcare intelligence assistant. You help clinical staff,
    administrators, and analysts explore patient data, claims, appointments,
    medical documents, and consultation recordings. You combine structured
    data analysis with unstructured text search across AI-processed medical
    reports (PDFs), text documents, and audio transcriptions.
    Provide concise, clinically relevant answers. When presenting financial
    data, include totals and averages where helpful. When referencing documents
    or audio transcripts, cite the file name. Always protect patient privacy
    in your responses.
  orchestration: >
    Use HealthcareAnalyst for questions about patients, providers, claims
    (billed amounts, denied claims, insurance plans), appointments (scheduling,
    visit types, durations), and any quantitative or aggregate questions.

    Use PDFSearch for questions about specific PDF medical document content -
    diagnoses, lab results, prescriptions, radiology findings, clinical notes,
    or any information found in PDF medical reports.

    Use TXTSearch for questions about text-based medical documents -
    clinical notes, intake forms, nursing notes, referral letters, or any
    information found in TXT medical files.

    Use AudioSearch for questions about patient consultation recordings -
    what was discussed in a visit, chief complaints, medications mentioned,
    follow-up actions, or any information from transcribed audio.

    If a question spans both structured and unstructured data, use
    HealthcareAnalyst for the structured part and the appropriate search tool
    for the unstructured part, then combine the results in your response.
  sample_questions:
    - question: "Which providers have the highest total billed amounts?"
    - question: "How many claims were denied and what were the reasons?"
    - question: "Show me patients on Medicare Advantage with cardiology appointments"
    - question: "Find PDF documents mentioning diabetes or hypertension"
    - question: "Search text documents for clinical notes about medication changes"
    - question: "What consultations discussed medication changes?"
    - question: "Summarize the consultation notes for follow-up visits"
    - question: "What is the average claim amount by specialty?"
    - question: "Find radiology reports with abnormal findings"
    - question: "Which patients have documents, text files, and audio recordings?"
    - question: "What are the most common diagnoses across all claims?"
    - question: "Compare findings across PDF and TXT documents for the same patient"

tools:
  - tool_spec:
      type: cortex_analyst_text_to_sql
      name: HealthcareAnalyst
      description: >
        Queries structured healthcare data including patients, providers,
        claims (billing, diagnosis codes, procedure codes, payment status),
        and appointments (scheduling, visit types, durations). Use for any
        quantitative questions about costs, claim denials, patient counts,
        appointment trends, insurance plans, and provider performance.
  - tool_spec:
      type: cortex_search
      name: PDFSearch
      description: >
        Searches AI-processed PDF medical documents including lab reports,
        discharge summaries, prescriptions, radiology reports, clinical notes,
        and insurance claims. PDFs have been parsed with AI_PARSE_DOCUMENT,
        enriched with AI_EXTRACT, AI_CLASSIFY, AI_SENTIMENT, AI_SUMMARIZE,
        AI_TRANSLATE, AI_REDACT, and AI_COMPLETE. Search across the full text,
        summaries, key insights, and extracted diagnoses.
  - tool_spec:
      type: cortex_search
      name: TXTSearch
      description: >
        Searches AI-processed text medical documents including clinical notes,
        patient intake forms, nursing notes, and referral letters. Text files
        have been enriched with AI_EXTRACT, AI_CLASSIFY, AI_SENTIMENT,
        AI_SUMMARIZE, AI_TRANSLATE, AI_REDACT, and AI_COMPLETE. Search across
        the full text, summaries, key insights, and extracted diagnoses.
  - tool_spec:
      type: cortex_search
      name: AudioSearch
      description: >
        Searches transcribed patient consultation recordings (WAV and MP3 audio).
        Consultations have been transcribed with AI_TRANSCRIBE and enriched
        with AI_EXTRACT, AI_CLASSIFY, AI_SENTIMENT, AI_SUMMARIZE, AI_TRANSLATE,
        and AI_COMPLETE (SOAP notes). Search across full transcripts, summaries,
        consultation notes, and extracted chief complaints.

tool_resources:
  HealthcareAnalyst:
    semantic_view: "HEALTHCARE_AI_DEMO.ANALYTICS.HEALTHCARE_ANALYTICS"
  PDFSearch:
    name: "HEALTHCARE_AI_DEMO.PROCESSED.PDF_SEARCH"
  TXTSearch:
    name: "HEALTHCARE_AI_DEMO.PROCESSED.TXT_SEARCH"
  AudioSearch:
    name: "HEALTHCARE_AI_DEMO.PROCESSED.AUDIO_SEARCH"
$$;

-----------------------------------------------------------------------
-- VERIFY
-----------------------------------------------------------------------
DESCRIBE AGENT ANALYTICS.HEALTHCARE_INTELLIGENCE_AGENT;

In [ ]:
DESCRIBE AGENT HEALTHCARE_AI_DEMO.ANALYTICS.HEALTHCARE_INTELLIGENCE_AGENT;

## Step 11: Test the Agent

**Recommended:** Go to **AI & ML → Intelligence → New Agent** → select `HEALTHCARE_AI_DEMO.ANALYTICS.HEALTHCARE_INTELLIGENCE_AGENT`

**Or test via SQL:**

In [ ]:
SELECT SNOWFLAKE.CORTEX.INVOKE_AGENT(
  'HEALTHCARE_AI_DEMO.ANALYTICS.HEALTHCARE_INTELLIGENCE_AGENT',
  'Which providers have the highest total billed amounts?'
);

In [ ]:
SELECT SNOWFLAKE.CORTEX.INVOKE_AGENT(
  'HEALTHCARE_AI_DEMO.ANALYTICS.HEALTHCARE_INTELLIGENCE_AGENT',
  'Find documents mentioning diabetes or hypertension'
);

In [ ]:
SELECT SNOWFLAKE.CORTEX.INVOKE_AGENT(
  'HEALTHCARE_AI_DEMO.ANALYTICS.HEALTHCARE_INTELLIGENCE_AGENT',
  'What consultations discussed medication changes?'
);

## Pipeline Summary

In [ ]:
SELECT 'FILES_LOG' AS OBJECT, COUNT(*) AS ROW_COUNT FROM HEALTHCARE_AI_DEMO.RAW.FILES_LOG
UNION ALL SELECT 'PDF_INTELLIGENCE', COUNT(*) FROM HEALTHCARE_AI_DEMO.PROCESSED.PDF_INTELLIGENCE
UNION ALL SELECT 'TXT_INTELLIGENCE', COUNT(*) FROM HEALTHCARE_AI_DEMO.PROCESSED.TXT_INTELLIGENCE
UNION ALL SELECT 'AUDIO_INTELLIGENCE', COUNT(*) FROM HEALTHCARE_AI_DEMO.PROCESSED.AUDIO_INTELLIGENCE
UNION ALL SELECT 'PROVIDERS', COUNT(*) FROM HEALTHCARE_AI_DEMO.ANALYTICS.PROVIDERS
UNION ALL SELECT 'PATIENTS', COUNT(*) FROM HEALTHCARE_AI_DEMO.ANALYTICS.PATIENTS
UNION ALL SELECT 'CLAIMS', COUNT(*) FROM HEALTHCARE_AI_DEMO.ANALYTICS.CLAIMS
UNION ALL SELECT 'APPOINTMENTS', COUNT(*) FROM HEALTHCARE_AI_DEMO.ANALYTICS.APPOINTMENTS;

## Cleanup

In [ ]:
-- Uncomment to remove all lab objects:
-- ALTER TASK HEALTHCARE_AI_DEMO.RAW.PROCESS_NEW_FILES_TASK SUSPEND;
-- DROP DATABASE IF EXISTS HEALTHCARE_AI_DEMO;
-- DROP WAREHOUSE IF EXISTS HEALTHCARE_AI_WH;
-- DROP STORAGE INTEGRATION IF EXISTS HEALTHCARE_S3_INTEGRATION;